### Regional Performance

In [0]:
%sql
CREATE OR REPLACE TABLE gold.region_performance
USING DELTA
AS
SELECT
  c.country,
  p.category,
 
  COUNT(DISTINCT c.customer_key)                    AS total_customers,
  COUNT(DISTINCT s.order_number)                    AS total_orders,
  SUM(s.quantity)                                   AS units_sold,
  ROUND(SUM(s.sales_amount), 2)                     AS total_revenue,
  ROUND(AVG(s.sales_amount), 2)                     AS avg_order_value,
 
  -- Each country's share of total global revenue
  ROUND(
    SUM(s.sales_amount) * 100.0
    / SUM(SUM(s.sales_amount)) OVER (), 2
  )                                                 AS global_revenue_share_pct,
 
  -- Revenue per customer — efficiency metric
  ROUND(
    SUM(s.sales_amount)
    / NULLIF(COUNT(DISTINCT c.customer_key), 0), 2
  )                                                 AS revenue_per_customer
 
FROM silver.sales s
JOIN silver.customers c  ON s.customer_key = c.customer_key
JOIN silver.products  p  ON s.product_key  = p.product_key
 
GROUP BY c.country, p.category
ORDER BY total_revenue DESC;



### Fulfilment Performance

Step 1: Order-Level Fulfilment Detail

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fulfilment_performance
USING DELTA
AS
SELECT
  s.order_number,
  c.country,
  s.order_date,
  s.shipping_date,
  s.due_date,
  s.days_to_ship,
 
  -- Was the order shipped on time?
  CASE
    WHEN s.shipping_date <= s.due_date THEN 'On Time'
    ELSE 'Late'
  END                                               AS delivery_status,
 
  -- How many days late (negative) or early (positive)?
  DATEDIFF(s.due_date, s.shipping_date)             AS days_early_or_late,
 
  s.sales_amount
 
FROM workspace.silver.sales s
JOIN workspace.silver.customers c
  ON s.customer_key = c.customer_key;



Step2: Country-Level Fulfilment Summary

In [0]:

%sql
CREATE OR REPLACE TABLE gold.fulfilment_summary
USING DELTA
AS
SELECT
  country,
  COUNT(*)                                          AS total_orders,
  ROUND(AVG(days_to_ship), 1)                       AS avg_days_to_ship,
 
  SUM(CASE WHEN delivery_status = 'On Time'
      THEN 1 ELSE 0 END)                            AS on_time_orders,
 
  SUM(CASE WHEN delivery_status = 'Late'
      THEN 1 ELSE 0 END)                            AS late_orders,
 
  -- On-time rate % — the KPI management tracks
  ROUND(
    SUM(CASE WHEN delivery_status = 'On Time' THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 2
  )                                                 AS on_time_rate_pct
 
FROM gold.fulfilment_performance
GROUP BY country
ORDER BY on_time_rate_pct ASC;